##### Copyright 2025 Perceptron AI.

In [ ]:
# Licensed under the MIT License (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://opensource.org/licenses/MIT
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Capability — Single-Image In-Context Learning (Perceptron Mk1)
Guide the model with one exemplar image before detecting the same object in a different scene.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/perceptron-ai-inc/perceptron/blob/main/cookbook/recipes/capabilities/perceptron-mk1/in-context-learning-image.ipynb)

## Install dependencies

In [ ]:
%pip install --upgrade perceptron --quiet

## Download assets

In [ ]:
ASSET_BASE = "https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets"

!curl -so cake_mixer_example.webp {ASSET_BASE}/in-context-learning/single/cake_mixer_example.webp
!curl -so find_kitchen_item.webp {ASSET_BASE}/in-context-learning/single/find_kitchen_item.webp

## Configure the Perceptron client
Authenticate once and point the SDK at Perceptron Mk1.

In [ ]:
import os
from pathlib import Path

from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

from perceptron import annotate_image, bbox, configure, detect, image

api_key = os.getenv("PERCEPTRON_API_KEY", "<your Perceptron API key>")
if not api_key or api_key.startswith("<"):
    raise RuntimeError("Set PERCEPTRON_API_KEY or replace the placeholder in this cell.")

configure(
    provider="perceptron",
    model="perceptron-mk1",
    api_key=api_key,
)

EXAMPLE_IMAGE = "cake_mixer_example.webp"
TARGET_IMAGE = "find_kitchen_item.webp"
ANNOTATED_PATH = Path("find_kitchen_item_annotated.png")

## Bootstrap the exemplar box
Run one detection pass on the exemplar image so we can feed a precise bounding box back as context.

In [ ]:
display(Image.open(EXAMPLE_IMAGE))
bootstrap = detect(
    image(str(EXAMPLE_IMAGE)),
    classes=["objectCategory1"],
    max_outputs=1,
)
if not bootstrap.boxes:
    raise RuntimeError("Detect returned no boxes for the exemplar. Adjust the label or image.")

first_box = bootstrap.boxes[0]
example_shot = annotate_image(
    str(EXAMPLE_IMAGE),
    {
        "objectCategory1": [
            bbox(
                int(first_box.top_left.x),
                int(first_box.top_left.y),
                int(first_box.bottom_right.x),
                int(first_box.bottom_right.y),
                mention="objectCategory1",
            )
        ]
    },
)
collections = example_shot.get("collections") or []
boxes = example_shot.get("boxes") or []
placeholder = "objectCategory1"
if collections:
    exemplar_annotation = collections[0]
elif boxes:
    exemplar_annotation = boxes[0]
else:
    raise RuntimeError("annotate_image returned no collections or boxes; check the exemplar annotations.")
print("Prepared exemplar guidance with", getattr(exemplar_annotation, "mention", placeholder) or placeholder)

## Detect the same object in a new scene
Pass the exemplar annotation back to `detect` via the `examples` argument to nudge the model toward consistent grounding.

In [ ]:
display(Image.open(TARGET_IMAGE))
result = detect(
    image(str(TARGET_IMAGE)),
    classes=["objectCategory1"],
    examples=[example_shot],
)

print(result.text)
boxes = result.boxes or []
print(f"Returned {len(boxes)} grounded regions")

## Render the grounded output
Overlay the predicted boxes on the target scene and preview the saved PNG inline.

In [ ]:
img = Image.open(TARGET_IMAGE).convert("RGB")
draw = ImageDraw.Draw(img)
try:
    font = ImageFont.truetype("arial.ttf", size=20)
except OSError:
    font = ImageFont.load_default()

if boxes:

    def to_px(point):
        return point.x / 1000 * img.width, point.y / 1000 * img.height

    for box in boxes:
        top_left = to_px(box.top_left)
        bottom_right = to_px(box.bottom_right)
        draw.rectangle([top_left, bottom_right], outline="lime", width=3)
        label = box.mention or getattr(box, "label", None) or "objectCategory1"
        text_position = (top_left[0], max(top_left[1] - 20, 0))
        draw.text(text_position, label, fill="lime", font=font)
else:
    print("No boxes returned; adjust the exemplar or label and rerun.")

img.save(ANNOTATED_PATH)
display(img)
print(f"Saved annotated target to {ANNOTATED_PATH}")

## Conclusion & next steps
- Swap in different exemplar / target pairs to explore other categories.
- Add more than one exemplar by appending to the `examples` list for tougher distinctions.
- For an open-ended (non-detection) ICL flow on a video, see [In-context learning (Video)](https://github.com/perceptron-ai-inc/perceptron/blob/main/cookbook/recipes/capabilities/perceptron-mk1/in-context-learning-video.ipynb).